# 最近进展与超额误差选模型

单机制、冻结四候选、严格成熟年度验证；已复用历史的诊断实验。新候选内层拟合为0，外层拟合最多11次。

控制：原平均两腿MSE选择。候选：SOXL-minus-SOXX H5收益差MSE选择。直接SOXX按相同窗口和费用比较。Qlib DatasetH、Recorder、原生Backtest和risk_analysis是证据入口。

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
ENTRY_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "AGENTS.md").is_file())
EVIDENCE_ROOT = Path(os.environ.get("SOXL_EVIDENCE_ROOT", ENTRY_ROOT)).resolve()
SOURCE_ROOT = Path(os.environ.get("SOXL_RESEARCH_SOURCE_ROOT", ENTRY_ROOT)).resolve()
sys.path.insert(0, str(SOURCE_ROOT))
CARD = ENTRY_ROOT / "research" / "benchmark_relative_loss_selection_20261012"
PLAN = json.loads((CARD / "plan.json").read_text())
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.workflow import R
from src.workflow.relative_loss_experiment import load_frozen_selection_inputs, run_relative_loss_experiment
qlib.init(provider_uri=str(EVIDENCE_ROOT / PLAN["providers"]["price"]["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(freq="day")
price_readback = D.features(["SOXL", "SOXX"], ["$open", "$close"], start_time="2026-09-21", end_time=PLAN["as_of"], freq="day")
assert not price_readback.empty
print({"provider_uri": PLAN["providers"]["price"]["provider_uri"], "region": "us", "as_of": PLAN["as_of"], "calendar_sessions": len(calendar)})
display(Markdown((CARD / "progress_review.md").read_text()))


In [ ]:
inputs = load_frozen_selection_inputs(evidence_root=EVIDENCE_ROOT, source_root=SOURCE_ROOT, plan=PLAN)
assert isinstance(inputs["bundle"]["control"], DatasetH)
prepared = inputs["bundle"]["control"].prepare("all", col_set=["feature", "label", "metadata"])
assert "information_value" not in prepared["feature"]
print({"native_dataset_rows": len(prepared), "features": len(prepared["feature"].columns), "parent_recorder": inputs["source_audit"]["recorder_id"], "source_read_only": inputs["source_audit"]["read_only"]})


In [ ]:
summary = run_relative_loss_experiment(evidence_root=EVIDENCE_ROOT, source_root=SOURCE_ROOT, card=CARD, plan_path=CARD / "plan.json", inputs=inputs)
print({k: summary[k] for k in ["recorder_id", "new_fit_calls", "changed_selection_years", "native_cases", "inherited_native_cases", "recomputed_native_cases", "historical_gate_pass", "incremental_information_pass", "incremental_portfolio_pass", "promotion_decision"]})


In [ ]:
from src.core.recorder_sources import load_verified_recorder_objects
objects, readback_audit = load_verified_recorder_objects(CARD, expected_recorder_id=summary["recorder_id"], artifacts=["aggregate/tables.pkl"])
tables = objects["aggregate/tables.pkl"]
display(tables["model_selection.csv"])
display(tables["backtest_benchmark_comparison.csv"])
display(tables["forecast_information.csv"])
display(tables["position_usage.csv"])
with R.uri_context(summary["recorder_uri"]):
    assert R.get_recorder(experiment_name=summary["experiment"], recorder_id=summary["recorder_id"]).status == "FINISHED"
display(Markdown((CARD / "conclusion.md").read_text()))


## 中文结论

待执行后同步本卡conclusion.md；未执行前不作科学结论。